# LeetCode 378 — Kth Smallest Element in a Sorted Matrix bằng Python

Link đề: https://leetcode.com/problems/kth-smallest-element-in-a-sorted-matrix/ · Độ khó: **Medium** · Chủ đề: Array, Binary Search, Matrix, Sorting, Heap (Priority Queue)

Skeleton LeetCode:

```python
class Solution:
    def kthSmallest(self, matrix: List[List[int]], k: int) -> int:
        ...
```

Ghi chú: đề được viết lại bằng tiếng Việt theo đề gốc trên LeetCode; mọi đáp án trong notebook đều đã được kiểm chứng bằng code (vét cạn và các oracle độc lập).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: ma trận `n x n` có **mỗi hàng và mỗi cột** tăng dần (không giảm); tìm phần tử nhỏ thứ `k` **theo thứ tự sắp xếp** (các giá trị trùng vẫn được tính riêng), với bộ nhớ tốt hơn `O(n²)`.
2. Nắm các công cụ Python: ma trận là list lồng nhau, `heapq` (`heappush`, `heappop`, `heapify`), tuple trong heap, `bisect`, vòng lặp **hai con trỏ** trên ma trận.
3. Đi từ **trải phẳng + sắp xếp** (`O(n² log n)` thời gian, `O(n²)` bộ nhớ) → **min-heap trộn `n` hàng đã sắp xếp** → **max-heap kích thước `k`** → **tìm kiếm nhị phân trên *giá trị*** với phép đếm "bậc thang" `O(n)`.
4. Hiểu mấu chốt: ma trận **không** sắp xếp toàn cục theo chỉ số phẳng, nên không binary search trên chỉ số; nhưng ta **có thể** binary search trên **khoảng giá trị** `[matrix[0][0], matrix[n-1][n-1]]` vì hàm "có bao nhiêu phần tử `<= mid`" là **đơn điệu**.
5. Tránh lỗi kinh điển: đếm sai biên (`<` thay `<=`), nhầm "phần tử thứ `k` phân biệt", trung điểm với số âm, đưa cả ma trận vào heap (vi phạm bộ nhớ), quên đẩy phần tử kế tiếp của hàng.

## 2. Đề bài & ví dụ

Cho ma trận `matrix` kích thước `n x n`, trong đó **mỗi hàng và mỗi cột** được sắp xếp theo thứ tự **tăng dần (không giảm)**. Trả về phần tử **nhỏ thứ `k`** trong ma trận.

Lưu ý: đó là phần tử thứ `k` **trong thứ tự đã sắp xếp** (các giá trị bằng nhau vẫn đếm nhiều lần), **không phải** phần tử thứ `k` *khác nhau*.

Bạn phải tìm lời giải có **độ phức tạp bộ nhớ tốt hơn `O(n²)`**.

**Ví dụ 1:** `matrix = [[1,5,9],[10,11,13],[12,13,15]]`, `k = 8` → `13`. (Các phần tử đã sắp xếp: `1,5,9,10,11,12,13,13,15`; phần tử thứ 8 là `13`.)

**Ví dụ 2:** `matrix = [[-5]]`, `k = 1` → `-5`.

**Ràng buộc:** `n == matrix.length == matrix[i].length`; `1 <= n <= 300`; `-10^9 <= matrix[i][j] <= 10^9`; mọi hàng và cột đều không giảm; `1 <= k <= n²`.

**Follow up:** có thể dùng bộ nhớ `O(1)`? Có thể `O(n)` thời gian? (Lời giải `O(n)` rất phức tạp, ngoài phạm vi phỏng vấn.)

In [1]:
import bisect
import heapq
import itertools
import random
import time
from typing import List

examples = [
    ([[1, 5, 9], [10, 11, 13], [12, 13, 15]], 8, 13),
    ([[-5]], 1, -5),
]
for matrix, k, expected in examples:
    print(matrix, "k =", k, "->", expected)

[[1, 5, 9], [10, 11, 13], [12, 13, 15]] k = 8 -> 13
[[-5]] k = 1 -> -5


## 3. Kiến thức nền

### 3.1. Ma trận = list của các list, và "trải phẳng"

`matrix[i][j]` là phần tử hàng `i`, cột `j`. Muốn thành **một dãy** để sắp xếp, ta **trải phẳng (flatten)**: `[x for row in matrix for x in row]` (đọc: "với mỗi hàng, với mỗi `x` trong hàng, lấy `x`").

In [2]:
matrix = [[1, 5, 9], [10, 11, 13], [12, 13, 15]]
phang = [x for row in matrix for x in row]
print(phang)                                  # [1, 5, 9, 10, 11, 13, 12, 13, 15]
print(sorted(phang))                          # [1, 5, 9, 10, 11, 12, 13, 13, 15]
print(len(matrix), len(matrix[0]))            # n = 3

[1, 5, 9, 10, 11, 13, 12, 13, 15]
[1, 5, 9, 10, 11, 12, 13, 13, 15]
3 3


Chú ý: trải phẳng `[1,5,9,10,11,13,12,13,15]` **không** tăng dần (`13` rồi `12`) — ma trận chỉ tăng theo **hàng** và **cột**, không tăng theo thứ tự đọc.

### 3.2. Heap (hàng đợi ưu tiên) với `heapq`

`heapq` trong Python là **min-heap**: phần tử nhỏ nhất luôn ở đầu và lấy ra nhanh.

- `heapq.heappush(h, x)`: thêm `x` — `O(log n)`.
- `heapq.heappop(h)`: lấy và bỏ phần tử nhỏ nhất — `O(log n)`.
- `h[0]`: xem phần tử nhỏ nhất — `O(1)`.
- `heapq.heapify(list)`: biến list thành heap — `O(n)`.
- Muốn **max-heap**: lưu **số đối** (`-x`).

In [3]:
h = []
for x in [5, 1, 9, 3]:
    heapq.heappush(h, x)
print("nhỏ nhất:", h[0])                       # 1
print([heapq.heappop(h) for _ in range(len(h))])   # [1, 3, 5, 9] - lấy ra theo thứ tự tăng

mh = []                                          # max-heap bằng số đối
for x in [5, 1, 9, 3]:
    heapq.heappush(mh, -x)
print("lớn nhất:", -mh[0])                       # 9

nhỏ nhất: 1
[1, 3, 5, 9]
lớn nhất: 9


### 3.3. Tuple trong heap

Heap so sánh phần tử bằng `<`. Với **tuple**, Python so sánh **từng phần tử từ trái sang phải**. Vậy `(giá_trị, hàng, cột)` được ưu tiên theo `giá_trị` trước; nếu bằng nhau thì xét `hàng`, rồi `cột`. Cách này cho ta biết một giá trị đến từ ô nào của ma trận.

In [4]:
h = []
heapq.heappush(h, (13, 2, 1))
heapq.heappush(h, (5, 0, 1))
heapq.heappush(h, (13, 1, 2))
print(heapq.heappop(h))                         # (5, 0, 1)
print(heapq.heappop(h))                         # (13, 1, 2)  - hai số 13: xét hàng nhỏ hơn

(5, 0, 1)
(13, 1, 2)


### 3.4. "Đếm số phần tử `<= x`" và tính đơn điệu

Định nghĩa `count(x)` = số phần tử của ma trận `<= x`. Khi `x` tăng thì `count(x)` **không giảm** (đơn điệu). Nếu `count(x) >= k` thì phần tử nhỏ thứ `k` nằm ở `<= x`; nếu `count(x) < k` thì nó nằm ở `> x`. Đây chính là điều kiện để **binary search trên giá trị**.

In [5]:
matrix = [[1, 5, 9], [10, 11, 13], [12, 13, 15]]
def count_slow(matrix, x):                       # đếm chậm O(n^2), chỉ để thí nghiệm
    return sum(1 for row in matrix for v in row if v <= x)

for x in range(0, 17, 2):
    print(f"count({x:>2}) = {count_slow(matrix, x)}")

count( 0) = 0
count( 2) = 1
count( 4) = 1
count( 6) = 2
count( 8) = 2
count(10) = 4
count(12) = 6
count(14) = 8
count(16) = 9


### 3.5. `bisect_right` — vị trí chèn bên phải

`bisect.bisect_right(a, x)` trên list **đã sắp xếp** cho số phần tử `<= x` (vị trí chèn ngay sau các giá trị bằng `x`). Mỗi hàng của ma trận đã sắp xếp, nên `bisect_right(row, x)` là số phần tử `<= x` trong hàng đó (`O(log n)`).

In [6]:
row = [10, 11, 13]
print(bisect.bisect_right(row, 12), bisect.bisect_right(row, 13), bisect.bisect_right(row, 9), bisect.bisect_right(row, 100))
# 2 3 0 3
print("count(12) =", sum(bisect.bisect_right(r, 12) for r in matrix))     # 6

2 3 0 3
count(12) = 6


### 3.6. Chia đôi số nguyên, kể cả số âm

`(lo + hi) // 2` là **phép chia sàn** (làm tròn xuống), đúng cho cả số âm trong Python (`-9 // 2 = -5`). Cẩn thận: `int((lo + hi) / 2)` **cắt về 0**, cho `-4` thay vì `-5` — dễ gây lặp vô hạn (mục 13).

In [7]:
print((-9) // 2, int(-9 / 2))       # -5 -4
print((3 + 8) // 2)                 # 5

-5 -4
5


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Ví dụ: `matrix = [[1,5,9],[10,11,13],[12,13,15]]`, `k = 8`.

**Cách ngây thơ:** liệt kê 9 số, sắp xếp: `1,5,9,10,11,12,13,13,15`, lấy số thứ 8 = `13`. Nhưng dùng `O(n²)` bộ nhớ — đề cấm.

**Quan sát 1 — số nhỏ nhất luôn ở góc trên trái:** `matrix[0][0] = 1`. Số nhỏ nhì phải là hàng xóm của nó: `matrix[0][1]` hoặc `matrix[1][0]` (vì mọi ô khác đều lớn hơn ít nhất một trong hai). Số nhỏ kế tiếp cũng nằm ở "biên" của vùng đã lấy. Đó chính là **trộn `n` hàng đã sắp xếp** — dùng heap.

**Quan sát 2 — đếm theo giá trị.** Thử `x = 12`: có bao nhiêu số `<= 12`? Hàng 0: `1,5,9` → 3. Hàng 1: `10,11` → 2. Hàng 2: `12` → 1. Tổng `6 < 8` → số thứ 8 phải **lớn hơn 12**. Thử `x = 13`: hàng 0: 3; hàng 1: 3; hàng 2: 2 → `8 >= 8` → số thứ 8 `<= 13`. Vậy đáp án là giá trị **nhỏ nhất `x` mà `count(x) >= 8`**: đó là `13`.

**Đếm nhanh nhờ cấu trúc ma trận ("bậc thang"):** bắt đầu ở góc **dưới trái**. Nếu `matrix[r][c] <= x` thì **cả cột `c` từ hàng `0` đến `r`** đều `<= x` (vì cột tăng dần xuống dưới; phần trên nhỏ hơn) → cộng `r + 1` rồi sang phải (`c += 1`). Nếu `matrix[r][c] > x` thì cả phần dưới của cột này cũng `> x` → lên hàng trên (`r -= 1`). Mỗi bước hoặc tăng `c` hoặc giảm `r` → tối đa `2n` bước: `O(n)`.

In [8]:
def count_le(matrix, x):
    """Số phần tử <= x, dùng 'bậc thang' từ góc dưới trái: O(n)."""
    n = len(matrix)
    r, c = n - 1, 0
    dem = 0
    while r >= 0 and c < n:
        if matrix[r][c] <= x:
            dem += r + 1            # cột c, từ hàng 0 đến r đều <= x
            c += 1
        else:
            r -= 1
    return dem

matrix = [[1, 5, 9], [10, 11, 13], [12, 13, 15]]
for x in range(0, 17, 2):
    assert count_le(matrix, x) == count_slow(matrix, x)
print("count(12) =", count_le(matrix, 12), "| count(13) =", count_le(matrix, 13))   # 6 8

count(12) = 6 | count(13) = 8


## 5. Cách 1 — Brute Force / baseline: trải phẳng rồi sắp xếp

**Ý tưởng:** đưa mọi phần tử vào một list, sắp xếp, lấy phần tử thứ `k` (chỉ số `k - 1`).

**Chạy ví dụ:**

In [9]:
def kth_sort(matrix, k):
    return sorted(x for row in matrix for x in row)[k - 1]

for matrix, k, expected in examples:
    got = kth_sort(matrix, k)
    print(got, expected)
    assert got == expected

13 13
-5 -5


**Độ phức tạp:** thời gian `O(n² log n)` (sắp xếp `n²` phần tử); bộ nhớ `O(n²)` — **vi phạm yêu cầu đề**, nhưng đúng và cực dễ; ta dùng làm **oracle**. (Trên LeetCode cách này vẫn được chấp nhận vì `n ≤ 300` → 90.000 phần tử, nhưng không đạt tinh thần đề.)

**Nút thắt:** ta bỏ qua cấu trúc sắp xếp sẵn của ma trận. Mỗi hàng **đã** tăng dần; chỉ cần "trộn" các hàng và dừng sau `k` bước.

## 6. Ý tưởng thuật toán / bước đột phá

1. **Trộn nhiều dãy đã sắp xếp (k-way merge):** `n` hàng đã sắp xếp. Giữ một heap chứa **phần tử hiện tại nhỏ nhất của từng hàng**. Lấy ra phần tử nhỏ nhất `k` lần; mỗi lần lấy xong thì đẩy phần tử **kế tiếp trong cùng hàng**. Phần tử lấy ra lần thứ `k` là đáp án. Bộ nhớ `O(n)`, thời gian `O(k log n)`.
2. **Binary search trên giá trị:** đáp án nằm trong `[matrix[0][0], matrix[n-1][n-1]]`. `count(mid)` đơn điệu, đếm được trong `O(n)`. Tìm giá trị nhỏ nhất có `count >= k` — đó luôn **là một phần tử của ma trận**. Bộ nhớ `O(1)`, thời gian `O(n log(max - min))`.
3. Hai hướng này khác nhau về *loại* ý tưởng: (1) thao tác trên **chỉ số/thứ hạng** bằng cách lấy lần lượt; (2) thao tác trên **không gian giá trị** bằng cách chia đôi.

## 7. Các cách tiếp theo

### Cách 2 — Min-heap trộn các hàng (k-way merge)

**Giải thích từng đoạn:**

- Khởi tạo heap với phần tử đầu **mỗi hàng**: `(matrix[r][0], r, 0)`. Chỉ cần `min(n, k)` hàng đầu (hàng thứ `k+1` trở đi không thể có phần tử nhỏ hơn đáp án — vì phần tử `matrix[r][0] >= matrix[0][0]` và cột tăng nên hàng `r` bị chặn bởi `r` phần tử phía trên cùng cột; ta chỉ cần `k` hàng đầu).
- Lặp `k - 1` lần: `heappop` lấy `(giá_trị, r, c)`; nếu `c + 1 < n` thì đẩy `(matrix[r][c+1], r, c+1)`.
- Sau `k - 1` lần bỏ, `heap[0][0]` là phần tử nhỏ thứ `k`.

In [10]:
def kth_minheap(matrix, k):
    n = len(matrix)
    heap = [(matrix[r][0], r, 0) for r in range(min(n, k))]
    heapq.heapify(heap)
    for _ in range(k - 1):
        val, r, c = heapq.heappop(heap)
        if c + 1 < n:
            heapq.heappush(heap, (matrix[r][c + 1], r, c + 1))
    return heap[0][0]

for matrix, k, expected in examples:
    got = kth_minheap(matrix, k)
    print(got, expected)
    assert got == expected

13 13
-5 -5


In [11]:
class SolutionMinHeap:
    def kthSmallest(self, matrix: List[List[int]], k: int) -> int:
        n = len(matrix)
        heap = [(matrix[r][0], r, 0) for r in range(min(n, k))]
        heapq.heapify(heap)
        for _ in range(k - 1):
            val, r, c = heapq.heappop(heap)
            if c + 1 < n:
                heapq.heappush(heap, (matrix[r][c + 1], r, c + 1))
        return heap[0][0]

print(SolutionMinHeap().kthSmallest([[1, 5, 9], [10, 11, 13], [12, 13, 15]], 8))

13


**Độ phức tạp:** thời gian `O(min(n,k) + k log min(n,k))`, tức `O(k log n)`; với `k ≤ n²` tệ nhất `O(n² log n)`. Bộ nhớ `O(min(n, k))` — tốt hơn `O(n²)`.

### Cách 3 — Max-heap kích thước `k` (giữ `k` phần tử nhỏ nhất)

**Ý tưởng khác:** duyệt mọi phần tử, **giữ `k` phần tử nhỏ nhất đã thấy** trong một **max-heap** (lưu số đối). Khi heap đã có `k` phần tử và gặp `x` nhỏ hơn phần tử **lớn nhất** trong heap, thay thế. Cuối cùng đỉnh max-heap = phần tử nhỏ thứ `k`.

**Giải thích từng đoạn:** `heap` chứa `-x`; `if len(heap) < k: push(-x)`; ngược lại nếu `x < -heap[0]` thì `heapreplace(heap, -x)` (pop + push một thao tác).

Ta tận dụng cấu trúc ma trận để **cắt sớm**: duyệt mỗi hàng và `break` khi `x >= -heap[0]` (các phần tử sau trong hàng lớn hơn nữa).

In [12]:
def kth_maxheap(matrix, k):
    heap = []                                   # lưu số đối -> max-heap
    for row in matrix:
        for x in row:
            if len(heap) < k:
                heapq.heappush(heap, -x)
            elif x < -heap[0]:
                heapq.heapreplace(heap, -x)     # bỏ phần tử lớn nhất, thêm x
            else:
                break                           # phần còn lại của hàng chỉ lớn hơn nữa
    return -heap[0]

for matrix, k, expected in examples:
    got = kth_maxheap(matrix, k)
    print(got, expected)
    assert got == expected

13 13
-5 -5


In [13]:
class SolutionMaxHeap:
    def kthSmallest(self, matrix: List[List[int]], k: int) -> int:
        heap = []
        for row in matrix:
            for x in row:
                if len(heap) < k:
                    heapq.heappush(heap, -x)
                elif x < -heap[0]:
                    heapq.heapreplace(heap, -x)
                else:
                    break
        return -heap[0]

print(SolutionMaxHeap().kthSmallest([[1, 5, 9], [10, 11, 13], [12, 13, 15]], 8))

13


**Độ phức tạp:** thời gian `O(n² log k)` trong trường hợp xấu (mỗi phần tử một thao tác heap `O(log k)`), thực tế nhanh hơn nhờ `break`; bộ nhớ `O(k)`. Với `k` gần `n²` thì không tốt hơn `O(n²)` về bộ nhớ — nên cách 2 và 4 mới là cách đúng tinh thần đề.

### Cách 4 — Binary search trên giá trị + đếm "bậc thang" (lời giải chuẩn, `O(1)` bộ nhớ)

**Giải thích từng đoạn:**

1. `lo = matrix[0][0]`, `hi = matrix[n-1][n-1]` — đáp án nằm trong đoạn này.
2. Trong khi `lo < hi`: `mid = (lo + hi) // 2`; tính `cnt = count_le(mid)`.
   - Nếu `cnt >= k`: đáp án `<= mid` → `hi = mid` (giữ `mid` vì có thể chính là đáp án).
   - Ngược lại: đáp án `> mid` → `lo = mid + 1`.
3. Khi `lo == hi`: đó là giá trị nhỏ nhất `x` có `count(x) >= k`. **Đó chắc chắn là một phần tử của ma trận** (xem mục 10).

In [14]:
def kth_binary(matrix, k):
    n = len(matrix)

    def count_le(x):
        r, c, dem = n - 1, 0, 0
        while r >= 0 and c < n:
            if matrix[r][c] <= x:
                dem += r + 1
                c += 1
            else:
                r -= 1
        return dem

    lo, hi = matrix[0][0], matrix[-1][-1]
    while lo < hi:
        mid = (lo + hi) // 2
        if count_le(mid) >= k:
            hi = mid
        else:
            lo = mid + 1
    return lo

for matrix, k, expected in examples:
    got = kth_binary(matrix, k)
    print(got, expected)
    assert got == expected

13 13
-5 -5


In [15]:
class SolutionBinary:
    def kthSmallest(self, matrix: List[List[int]], k: int) -> int:
        n = len(matrix)

        def count_le(x: int) -> int:
            r, c, dem = n - 1, 0, 0            # bắt đầu từ góc dưới trái
            while r >= 0 and c < n:
                if matrix[r][c] <= x:
                    dem += r + 1               # cả cột c từ hàng 0..r đều <= x
                    c += 1
                else:
                    r -= 1
            return dem

        lo, hi = matrix[0][0], matrix[-1][-1]
        while lo < hi:
            mid = (lo + hi) // 2
            if count_le(mid) >= k:
                hi = mid
            else:
                lo = mid + 1
        return lo

print(SolutionBinary().kthSmallest([[1, 5, 9], [10, 11, 13], [12, 13, 15]], 8))

13


**Độ phức tạp:** mỗi lần đếm `O(n)`; số vòng lặp `log₂(max - min) ≤ log₂(2·10⁹) ≈ 31`. Tổng `O(n · log(max - min))`, bộ nhớ `O(1)`. Với `n = 300`: ~`300 × 31 × 2 ≈ 20.000` phép — gần như tức thì.

## 8. TRACE từng bước

### 8.1. Trace Cách 4 (binary search) với `k = 8`

| vòng | lo | hi | mid | count(mid) | so với k=8 | cập nhật |
|---|---|---|---|---|---|---|
| 1 | 1 | 15 | 8 | 2 (`1,5`) | `< 8` | `lo = 9` |
| 2 | 9 | 15 | 12 | 6 | `< 8` | `lo = 13` |
| 3 | 13 | 15 | 14 | 8 | `>= 8` | `hi = 14` |
| 4 | 13 | 14 | 13 | 8 | `>= 8` | `hi = 13` |
| — | 13 | 13 | | | `lo == hi` | đáp án `13` |

In [16]:
def trace_binary(matrix, k):
    n = len(matrix)
    lo, hi = matrix[0][0], matrix[-1][-1]
    vong = 0
    print(f"{'vòng':>4} {'lo':>4} {'hi':>4} {'mid':>4} {'count':>6}  quyết định")
    while lo < hi:
        vong += 1
        mid = (lo + hi) // 2
        cnt = count_le(matrix, mid)
        print(f"{vong:>4} {lo:>4} {hi:>4} {mid:>4} {cnt:>6}  ", end="")      # in lo, hi TRƯỚC khi cập nhật
        if cnt >= k:
            print(f"cnt >= k -> hi = {mid}")
            hi = mid
        else:
            print(f"cnt <  k -> lo = {mid + 1}")
            lo = mid + 1
    print("Đáp án:", lo)
    return lo

assert trace_binary([[1, 5, 9], [10, 11, 13], [12, 13, 15]], 8) == 13

vòng   lo   hi  mid  count  quyết định
   1    1   15    8      2  cnt <  k -> lo = 9
   2    9   15   12      6  cnt <  k -> lo = 13
   3   13   15   14      8  cnt >= k -> hi = 14
   4   13   14   13      8  cnt >= k -> hi = 13
Đáp án: 13


### 8.2. Trace phép đếm "bậc thang" `count_le(x = 12)`

Xuất phát `(r, c) = (2, 0)`: `matrix[2][0] = 12 <= 12` → cộng `r + 1 = 3`, `c = 1`. `matrix[2][1] = 13 > 12` → `r = 1`. `matrix[1][1] = 11 <= 12` → cộng `2`, `c = 2`. `matrix[1][2] = 13 > 12` → `r = 0`. `matrix[0][2] = 9 <= 12` → cộng `1`, `c = 3` (kết thúc). Tổng `3 + 2 + 1 = 6`. Hàm sau in đúng quá trình đó.

In [17]:
def trace_count(matrix, x):
    n = len(matrix)
    r, c, dem = n - 1, 0, 0
    while r >= 0 and c < n:
        v = matrix[r][c]
        if v <= x:
            dem += r + 1
            print(f"(r={r}, c={c}) matrix={v:>2} <= {x}: cộng {r + 1} (cả cột {c}, hàng 0..{r}) -> dem={dem}, sang phải")
            c += 1
        else:
            print(f"(r={r}, c={c}) matrix={v:>2} >  {x}: lên hàng trên")
            r -= 1
    print("count =", dem)
    return dem

assert trace_count([[1, 5, 9], [10, 11, 13], [12, 13, 15]], 12) == 6

(r=2, c=0) matrix=12 <= 12: cộng 3 (cả cột 0, hàng 0..2) -> dem=3, sang phải
(r=2, c=1) matrix=13 >  12: lên hàng trên
(r=1, c=1) matrix=11 <= 12: cộng 2 (cả cột 1, hàng 0..1) -> dem=5, sang phải
(r=1, c=2) matrix=13 >  12: lên hàng trên
(r=0, c=2) matrix= 9 <= 12: cộng 1 (cả cột 2, hàng 0..0) -> dem=6, sang phải
count = 6


### 8.3. Trace Cách 2 (min-heap) với `k = 8`

Hàm in từng lần `heappop` và phần tử được đẩy thêm.

In [18]:
def trace_minheap(matrix, k):
    n = len(matrix)
    heap = [(matrix[r][0], r, 0) for r in range(min(n, k))]
    heapq.heapify(heap)
    print("heap ban đầu:", sorted(heap))
    for lan in range(1, k):
        val, r, c = heapq.heappop(heap)
        mo_ta = f"lần {lan}: lấy {val} (hàng {r}, cột {c})"
        if c + 1 < n:
            heapq.heappush(heap, (matrix[r][c + 1], r, c + 1))
            mo_ta += f" -> đẩy {matrix[r][c + 1]}"
        print(mo_ta, "| heap:", sorted(heap))
    print(f"Phần tử nhỏ thứ {k} = {heap[0][0]}")
    return heap[0][0]

assert trace_minheap([[1, 5, 9], [10, 11, 13], [12, 13, 15]], 8) == 13

heap ban đầu: [(1, 0, 0), (10, 1, 0), (12, 2, 0)]
lần 1: lấy 1 (hàng 0, cột 0) -> đẩy 5 | heap: [(5, 0, 1), (10, 1, 0), (12, 2, 0)]
lần 2: lấy 5 (hàng 0, cột 1) -> đẩy 9 | heap: [(9, 0, 2), (10, 1, 0), (12, 2, 0)]
lần 3: lấy 9 (hàng 0, cột 2) | heap: [(10, 1, 0), (12, 2, 0)]
lần 4: lấy 10 (hàng 1, cột 0) -> đẩy 11 | heap: [(11, 1, 1), (12, 2, 0)]
lần 5: lấy 11 (hàng 1, cột 1) -> đẩy 13 | heap: [(12, 2, 0), (13, 1, 2)]
lần 6: lấy 12 (hàng 2, cột 0) -> đẩy 13 | heap: [(13, 1, 2), (13, 2, 1)]
lần 7: lấy 13 (hàng 1, cột 2) | heap: [(13, 2, 1)]
Phần tử nhỏ thứ 8 = 13


## 9. Code nộp LeetCode

Ta nộp Cách 4 (binary search trên giá trị): `O(1)` bộ nhớ, trả lời follow up.

In [19]:
class Solution:
    def kthSmallest(self, matrix: List[List[int]], k: int) -> int:
        n = len(matrix)

        def count_le(x: int) -> int:
            # đếm số phần tử <= x bằng "bậc thang" từ góc dưới trái
            r, c, dem = n - 1, 0, 0
            while r >= 0 and c < n:
                if matrix[r][c] <= x:
                    dem += r + 1
                    c += 1
                else:
                    r -= 1
            return dem

        lo, hi = matrix[0][0], matrix[-1][-1]
        while lo < hi:
            mid = (lo + hi) // 2
            if count_le(mid) >= k:
                hi = mid
            else:
                lo = mid + 1
        return lo

sol = Solution()
for matrix, k, expected in examples:
    got = sol.kthSmallest(matrix, k)
    print(got, expected, "OK" if got == expected else "SAI")
    assert got == expected

13 13 OK
-5 -5 OK


## 10. Vì sao thuật toán đúng

**Cách 4 (binary search).**

1. **Đơn điệu:** `count(x)` (số phần tử `<= x`) không giảm khi `x` tăng. Tập `{x : count(x) >= k}` là một **nửa đường thẳng** `[x*, +∞)`. Đáp án (phần tử nhỏ thứ `k`) chính là `x*`:
   - `count(đáp án) >= k` vì có ít nhất `k` phần tử `<=` đáp án (chính `k` phần tử nhỏ nhất);
   - `count(đáp án - 1) < k`, vì phần tử nhỏ thứ `k` lớn hơn `đáp án - 1` nên chỉ có tối đa `k - 1` phần tử nhỏ hơn nó... (trường hợp trùng giá trị: các phần tử trùng đáp án vẫn `> đáp án - 1`).
2. **`x*` là một phần tử của ma trận.** Nếu `x*` không có trong ma trận thì `count(x*) = count(x* - 1)` (không phần tử nào bằng `x*` để thêm vào), mà `count(x* - 1) < k <= count(x*)` — mâu thuẫn.
3. **Khoảng tìm kiếm hợp lệ:** `x* ∈ [matrix[0][0], matrix[n-1][n-1]]` vì `k >= 1` (cần ít nhất phần tử nhỏ nhất) và `k <= n²` (phần tử lớn nhất đã đủ).
4. **Vòng lặp:** invariant `lo <= x* <= hi`. Nếu `count(mid) >= k` thì `mid >= x*` ⇒ `hi = mid` giữ invariant; ngược lại `mid < x*` ⇒ `lo = mid + 1` giữ invariant. Khoảng thu hẹp mỗi vòng nên dừng với `lo == hi == x*`.
5. **Phép đếm bậc thang đúng:** tại `(r, c)`, nếu `matrix[r][c] <= x` thì mọi ô `(r', c)` với `r' <= r` đều `<= matrix[r][c] <= x` (cột tăng) → cộng `r + 1`; các ô cột `c` ở hàng `> r` đã được đếm/loại ở các bước trước vì ta chỉ đi lên khi `matrix[r+1][c] > x`. Nên đếm đủ và không trùng.

**Cách 2 (min-heap).** Mỗi lần lấy ra phần tử nhỏ nhất trong heap; đó là phần tử nhỏ nhất **chưa lấy** của toàn ma trận vì với mỗi hàng, phần tử kế tiếp chưa lấy của hàng chính là phần tử nhỏ nhất của hàng trong số những phần tử chưa lấy, và heap chứa đại diện của **mọi** hàng (ít nhất là `k` hàng đầu; hàng thứ `i ≥ k` có phần tử đầu lớn hơn hoặc bằng `i` phần tử phía trên cùng cột nên không thể nằm trong top-`k`).

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Trải phẳng + sort | `O(n² log n)` | `O(n²)` |
| 2. Min-heap trộn hàng | `O(k log min(n,k))` | `O(min(n,k))` |
| 3. Max-heap kích thước `k` | `O(n² log k)` | `O(k)` |
| 4. Binary search giá trị | `O(n log(max-min))` | `O(1)` |

Với `n = 300`: trải phẳng 90.000 phần tử; binary search chỉ cỡ `300 × 31 × 2 ≈ 18.600` bước.

In [20]:
def tao_ma_tran(n, rng, buoc=5, goc=None):
    """Sinh ma trận n x n có hàng và cột không giảm (cho phép giá trị trùng)."""
    m = [[0] * n for _ in range(n)]
    m[0][0] = rng.randint(-10 ** 9, 10 ** 9 - 10 ** 6 * 2 * n) if goc is None else goc
    for i in range(n):
        for j in range(n):
            if i == 0 and j == 0:
                continue
            base = max(m[i - 1][j] if i else -10 ** 18, m[i][j - 1] if j else -10 ** 18)
            m[i][j] = base + rng.randint(0, buoc)
    return m

rng = random.Random(378)
lon = tao_ma_tran(300, rng, buoc=3000)
print("giá trị trong [%d, %d]" % (lon[0][0], lon[-1][-1]))
k = 45000
for ten, cls in [("sort", None), ("min-heap", SolutionMinHeap), ("max-heap", SolutionMaxHeap), ("binary", SolutionBinary)]:
    t = time.perf_counter()
    kq = kth_sort(lon, k) if cls is None else cls().kthSmallest(lon, k)
    print(f"{ten:<9}: kq = {kq}, {1000 * (time.perf_counter() - t):.2f} ms")

giá trị trong [-951183876, -949852170]
sort     : kq = -950553608, 11.78 ms
min-heap : kq = -950553608, 14.85 ms
max-heap : kq = -950553608, 13.58 ms
binary   : kq = -950553608, 1.50 ms


## 12. Bộ test

### 12.1. Ví dụ đề và các trường hợp biên

In [21]:
tests = [
    ([[1, 5, 9], [10, 11, 13], [12, 13, 15]], 8, 13),
    ([[-5]], 1, -5),
    ([[1, 2], [1, 3]], 2, 1),                      # trùng giá trị: phần tử thứ 2 là 1 (không phải 2)
    ([[1, 2], [1, 3]], 3, 2),
    ([[1, 1], [1, 1]], 4, 1),                      # toàn bằng nhau
    ([[1, 2], [3, 4]], 1, 1),                      # k = 1
    ([[1, 2], [3, 4]], 4, 4),                      # k = n^2
    ([[-10 ** 9, -10 ** 9 + 1], [-10 ** 9 + 2, 10 ** 9]], 4, 10 ** 9),   # biên giá trị
    ([[-3, -2], [-2, -1]], 2, -2),                 # số âm
    ([[1, 3, 5], [6, 7, 12], [11, 14, 14]], 6, 11),
]
for matrix, k, expected in tests:
    for fn in (kth_sort, kth_minheap, kth_maxheap, kth_binary, Solution().kthSmallest):
        got = fn(matrix, k)
        assert got == expected, (fn, matrix, k, got, expected)
print("OK", len(tests), "test x 5 cách")

OK

 10 test x 5 cách


### 12.2. Phản ví dụ cho các cách làm sai

1. **Lấy phần tử thứ `k` theo thứ tự đọc của ma trận trải phẳng** (hoặc binary search trên chỉ số phẳng): ma trận trải phẳng không sắp xếp (`13` rồi `12`); với `k = 7` kết quả sai.
2. **Đếm phần tử phân biệt** thay vì đếm cả trùng: `[[1,2],[1,3]]`, `k = 2` đúng là `1`; "phân biệt" cho `2`.

In [22]:
def sai_phan_biet(matrix, k):
    return sorted(set(x for row in matrix for x in row))[k - 1]

def sai_chi_so_phang(matrix, k):
    # SAI: coi ma trận trải phẳng đã sắp xếp nên chỉ cần lấy phần tử thứ k theo thứ tự đọc
    return [x for row in matrix for x in row][k - 1]

m1 = [[1, 2], [1, 3]]
m2 = [[1, 5, 9], [10, 11, 13], [12, 13, 15]]
print("phân biệt:", sai_phan_biet(m1, 2), "| đúng:", Solution().kthSmallest(m1, 2))
print("chỉ số phẳng (k=7):", sai_chi_so_phang(m2, 7), "| đúng:", Solution().kthSmallest(m2, 7))
assert sai_phan_biet(m1, 2) != Solution().kthSmallest(m1, 2)
assert sai_chi_so_phang(m2, 7) != Solution().kthSmallest(m2, 7)

phân biệt: 2 | đúng: 1
chỉ số phẳng (k=7): 12 | đúng: 13


### 12.3. Đối chiếu ngẫu nhiên (có seed) — 4 cách + oracle độc lập

Oracle độc lập thứ hai: `heapq.nsmallest(k, ...)[-1]` (thư viện), và phép đếm `bisect_right` thay cho bậc thang.

In [23]:
def kth_oracle(matrix, k):
    return heapq.nsmallest(k, (x for row in matrix for x in row))[-1]

def count_bisect(matrix, x):
    return sum(bisect.bisect_right(row, x) for row in matrix)

rng = random.Random(3780)
dem = 0
for _ in range(600):
    n = rng.randint(1, 8)
    m = tao_ma_tran(n, rng, buoc=rng.choice([0, 1, 2, 5, 50]), goc=rng.randint(-20, 20))
    for i in range(n):                                     # kiểm tra ma trận sinh ra hợp lệ
        for j in range(n):
            if i: assert m[i - 1][j] <= m[i][j]
            if j: assert m[i][j - 1] <= m[i][j]
    k = rng.randint(1, n * n)
    ket_qua = Solution().kthSmallest(m, k)
    assert ket_qua == kth_sort(m, k) == kth_oracle(m, k)
    assert ket_qua == kth_minheap(m, k) == kth_maxheap(m, k) == kth_binary(m, k)
    # phép đếm bậc thang khớp bisect trên mọi x trong khoảng giá trị
    for x in range(m[0][0] - 1, m[-1][-1] + 2):
        assert count_le(m, x) == count_bisect(m, x)
    dem += 1
print("4 cách + sort + nsmallest + 2 phép đếm khớp trên", dem, "ma trận ngẫu nhiên nhỏ")

4 cách + sort + nsmallest + 2 phép đếm khớp trên

 600 ma trận ngẫu nhiên nhỏ


In [24]:
rng = random.Random(37800)
for _ in range(40):
    n = rng.randint(50, 300)
    m = tao_ma_tran(n, rng, buoc=rng.choice([0, 10, 10 ** 4]))
    k = rng.randint(1, n * n)
    kq = Solution().kthSmallest(m, k)
    assert kq == kth_sort(m, k) == kth_minheap(m, k) == kth_maxheap(m, k), (n, k)
print("Đúng trên 40 ma trận lớn (n tới 300), giá trị trong khoảng ±1e9")

Đúng trên 40 ma trận lớn (n tới 300), giá trị trong khoảng ±1e9


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Binary search trên **chỉ số** của ma trận trải phẳng

Ma trận sắp xếp theo hàng và theo cột, **không** sắp xếp theo thứ tự đọc. Không thể chia đôi theo chỉ số.

In [25]:
m = [[1, 5, 9], [10, 11, 13], [12, 13, 15]]
phang = [x for row in m for x in row]
print(phang, "-> có tăng dần không?", phang == sorted(phang))
print("phần tử thứ 7 theo thứ tự đọc:", phang[6], "| đúng:", Solution().kthSmallest(m, 7))

[1, 5, 9, 10, 11, 13, 12, 13, 15] -> có tăng dần không? False
phần tử thứ 7 theo thứ tự đọc: 12 | đúng: 13


Vì sao sai: `phang[6] = 12` xuất hiện **sau** `phang[5] = 13`, nên thứ tự đọc không phải thứ tự sắp xếp. **Đúng:** binary search trên **giá trị** với `count(mid)`.

### Lỗi 2: Đếm sai biên — dùng `<` thay vì `<=`

`count(mid)` phải đếm phần tử `<= mid`. Nếu đếm `< mid` thì điều kiện `count >= k` bị lệch.

In [26]:
def kth_binary_sai_bien(matrix, k):
    n = len(matrix)
    def count_lt(x):                       # SAI: đếm phần tử < x
        r, c, dem = n - 1, 0, 0
        while r >= 0 and c < n:
            if matrix[r][c] < x:
                dem += r + 1
                c += 1
            else:
                r -= 1
        return dem
    lo, hi = matrix[0][0], matrix[-1][-1]
    while lo < hi:
        mid = (lo + hi) // 2
        if count_lt(mid) >= k:
            hi = mid
        else:
            lo = mid + 1
    return lo

m = [[1, 5, 9], [10, 11, 13], [12, 13, 15]]
print("sai:", kth_binary_sai_bien(m, 8), "| đúng:", Solution().kthSmallest(m, 8))

sai: 14 | đúng: 13


**Đúng:** `<=`, hoặc nếu đếm `<` thì phải đổi điều kiện thành `count_lt(mid) > k - 1`... dễ nhầm nên giữ `<=` với `>= k`.

### Lỗi 3: Trung điểm cắt về 0 khi có số âm → lặp vô hạn

`int((lo + hi) / 2)` làm tròn **về 0**, với số âm cho kết quả lớn hơn `//`. Khi `lo = -5`, `hi = -4`: `int(-4.5) = -4 = hi`, nếu `count(-4) >= k` thì `hi = -4` không đổi → **vòng lặp vô hạn**.

In [27]:
def kth_binary_sai_mid(matrix, k, gioi_han=100):
    n = len(matrix)
    lo, hi = matrix[0][0], matrix[-1][-1]
    vong = 0
    while lo < hi:
        vong += 1
        if vong > gioi_han:
            return "LẶP VÔ HẠN (dừng thủ công)"
        mid = int((lo + hi) / 2)               # SAI với số âm: cắt về 0
        if count_le(matrix, mid) >= k:
            hi = mid
        else:
            lo = mid + 1
    return lo

m = [[-5, -4], [-4, -3]]
print("int():", kth_binary_sai_mid(m, 2), "| đúng:", Solution().kthSmallest(m, 2))

int(): LẶP VÔ HẠN (dừng thủ công) | đúng: -4


**Đúng:** dùng `//` (làm tròn xuống) hoặc `lo + (hi - lo) // 2` — trong Python `//` đã đúng cho số âm.

### Lỗi 4: Đưa **cả ma trận** vào heap

Đẩy `n²` phần tử vào heap thì đúng nhưng bộ nhớ `O(n²)` — vi phạm đề. Chỉ nên đẩy **phần tử đầu mỗi hàng** rồi đẩy tiếp phần tử kế khi lấy ra.

### Lỗi 5: Quên đẩy phần tử kế tiếp của hàng sau khi `heappop`

In [28]:
def kth_minheap_sai(matrix, k):
    n = len(matrix)
    heap = [(matrix[r][0], r, 0) for r in range(n)]
    heapq.heapify(heap)
    for _ in range(k - 1):
        val, r, c = heapq.heappop(heap)       # SAI: quên đẩy matrix[r][c+1]
    return heap[0][0]

m = [[1, 5, 9], [10, 11, 13], [12, 13, 15]]
print("sai (k=3):", kth_minheap_sai(m, 3), "| đúng:", Solution().kthSmallest(m, 3))

sai (k=3): 12 | đúng: 9


Với `k = 3` đúng là `9` nhưng bản sai ra `12`: các phần tử `5, 9` không bao giờ vào heap (và với `k` lớn hơn, heap cạn rồi `heap[0]` còn gây `IndexError`/kết quả vô nghĩa). **Đúng:** sau mỗi `heappop`, nếu còn cột kế tiếp thì `heappush` phần tử kế tiếp của cùng hàng.

## 14. Mẹo & nhận dạng pattern

- **Từ khóa:** "phần tử nhỏ thứ k", "mỗi hàng/cột sắp xếp" → thử **k-way merge (heap)** hoặc **binary search trên giá trị**.
- **Binary search trên đáp án (không phải trên chỉ số):** khi có hàm **đơn điệu** `count(x)` đếm nhanh được. Dạng: "tìm `x` nhỏ nhất sao cho `điều_kiện(x)`". Template: `lo, hi = min, max; while lo < hi: mid = (lo+hi)//2; if ok(mid): hi = mid else lo = mid + 1; return lo`.
- **Đếm bậc thang (staircase)** trên ma trận sắp xếp: bắt đầu **góc dưới trái** (hoặc trên phải), mỗi bước loại một hàng hoặc một cột → `O(n)`. Cũng dùng trong *Search a 2D Matrix II* (240).
- **Khi nào dùng heap, khi nào binary search:** `k` nhỏ so với `n²` → heap `O(k log n)`; `k` lớn / yêu cầu `O(1)` bộ nhớ → binary search.
- **Họ bài:** *Find K Pairs with Smallest Sums* (373), *Kth Smallest Number in Multiplication Table* (668), *Find K-th Smallest Pair Distance* (719), *Median of Row Wise Sorted Matrix*, *Merge k Sorted Lists* (23), *Ugly Number II* (264).
- **Mẹo đếm trùng:** nhớ rằng "thứ tự sắp xếp" tính cả giá trị trùng; binary search trên giá trị xử lý tự nhiên vì `count(x)` đếm cả trùng.

## 15. Biến thể

### 15.1. Chỉ sắp xếp theo hàng (cột không sắp xếp)

Khi chỉ mỗi **hàng** tăng dần (cột tùy ý), phép đếm bậc thang không còn đúng. Nhưng `count(x)` vẫn có thể đếm bằng `bisect_right` trên từng hàng: `O(n log n)` mỗi lần, binary search trên giá trị vẫn chạy được, tổng `O(n log n log(max-min))`. Min-heap trộn hàng cũng vẫn đúng.

In [29]:
def kth_chi_hang_sap_xep(matrix, k):
    lo = min(row[0] for row in matrix)
    hi = max(row[-1] for row in matrix)
    while lo < hi:
        mid = (lo + hi) // 2
        if sum(bisect.bisect_right(row, mid) for row in matrix) >= k:
            hi = mid
        else:
            lo = mid + 1
    return lo

rng = random.Random(1)
for _ in range(300):
    n = rng.randint(1, 6)
    m = [sorted(rng.randint(-20, 20) for _ in range(n)) for _ in range(n)]    # cột KHÔNG sắp xếp
    k = rng.randint(1, n * n)
    assert kth_chi_hang_sap_xep(m, k) == kth_sort(m, k)
print("Binary search + bisect đúng cả khi chỉ hàng được sắp xếp")

Binary search + bisect đúng cả khi chỉ hàng được sắp xếp


### 15.2. Ma trận chữ nhật `m x n`

Đổi giới hạn vòng đếm (`r = m - 1`, `c < n`), `hi = matrix[-1][-1]`; mọi thứ khác giữ nguyên.

### 15.3. Kth Smallest Number in Multiplication Table (LeetCode 668)

Bảng `m x n` với `a[i][j] = i * j` (chỉ số từ `1`) cũng sắp xếp theo hàng và cột, nhưng `m, n` rất lớn → không thể dựng. May là `count(x) = Σ_i min(x // i, n)` đếm trực tiếp → binary search trên giá trị. Ta làm ở bài tập 5.

### 15.4. Các biến thể khác

- **Trung vị** của ma trận sắp xếp hàng: `k = (n² + 1) / 2`.
- **Kth Smallest Pair Distance (719)**, **Kth Smallest Sum of Matrix With Sorted Rows (1439)**.
- **Dòng dữ liệu (stream):** giữ max-heap kích thước `k`.
- **Cách `O(n)` (Frederickson–Johnson):** khó, ngoài phạm vi.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Trải phẳng + sort | Liệt kê hết rồi sort | `O(n² log n)` | `O(n²)` | Oracle kiểm tra, `n` nhỏ |
| 2. Min-heap trộn hàng | k-way merge, lấy `k` lần | `O(k log n)` | `O(n)` | `k` nhỏ; ý tưởng "trộn" quen thuộc |
| 3. Max-heap kích thước `k` | Giữ `k` nhỏ nhất | `O(n² log k)` | `O(k)` | Dữ liệu dạng stream, `k` nhỏ |
| 4. Binary search giá trị | `count(mid)` đơn điệu | `O(n log(max-min))` | `O(1)` | **Nộp bài**, bộ nhớ nhỏ nhất |

In [30]:
rng = random.Random(7)
m = tao_ma_tran(300, rng, buoc=3000)
for k in (1, 300, 45000, 90000):
    ket = []
    for ten, fn in [("min-heap", kth_minheap), ("max-heap", kth_maxheap), ("binary", kth_binary)]:
        t = time.perf_counter()
        fn(m, k)
        ket.append(f"{ten}={1000 * (time.perf_counter() - t):7.2f}ms")
    print(f"k={k:>5}: " + "  ".join(ket))

k=    1: min-heap=   0.01ms  max-heap=   0.14ms  binary=   0.83ms
k=  300: min-heap=   0.16ms  max-heap=   0.21ms  binary=   0.66ms
k=45000: min-heap=  17.10ms  max-heap=  11.52ms  binary=   1.50ms
k=90000: min-heap=  30.85ms  max-heap=  12.90ms  binary=   0.85ms


Quan sát: min-heap rất nhanh khi `k` nhỏ nhưng chậm dần theo `k`; binary search gần như không đổi theo `k`.

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: (a) `bisect.bisect_right([2, 4, 4, 7], 4)`; (b) `count_le([[1,2],[3,4]], 2)`; (c) `(−7 + 2) // 2`; (d) `kthSmallest([[1,2],[1,3]], 3)`; (e) `heapq.heappop` đầu tiên trên heap `[(5,0), (3,1), (3,0)]` sau `heapify`.

**Bài 2 (tính tay).** Với `matrix = [[1,3,5],[6,7,12],[11,14,14]]`, `k = 6`: tính `count(7)`, `count(10)`, `count(11)` bằng bậc thang (ghi từng bước `(r, c)`), từ đó suy ra đáp án (bằng `11`).

**Bài 3 (điền TODO).** Hoàn thành `count_todo(matrix, x)` (đếm số phần tử `<= x` bằng bậc thang).

In [31]:
def count_todo(matrix, x):
    # TODO 1: n = len(matrix); r = n - 1; c = 0; dem = 0
    # TODO 2: while r >= 0 and c < n:
    #            nếu matrix[r][c] <= x: dem += r + 1; c += 1
    #            ngược lại: r -= 1
    # TODO 3: return dem
    return None

# KHÔNG gọi count_todo ở đây để "Run All" không lỗi (chưa điền xong).

**Bài 4 (tự viết lại).** Viết `kth_heap2(matrix, k)` bằng min-heap trộn hàng **mà không nhìn lại** mục 7. Kiểm tra với ví dụ đề (`→ 13`).

**Bài 5 (biến thể / mini challenge).** **Kth Smallest Number in Multiplication Table (668)**: cho `m, n, k`, bảng `a[i][j] = i * j` với `1 <= i <= m`, `1 <= j <= n`. Viết `kth_mul_table(m, n, k)` bằng binary search; kiểm tra `(3, 3, 5) → 3`, `(2, 3, 6) → 6`. Gợi ý: `count(x) = Σ_{i=1..m} min(x // i, n)`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: (a) `3`, (b) `2`, (c) `-3` (vì `-5 // 2 = -3`), (d) `2`, (e) `(3, 0)`. Bài 3: bắt đầu từ góc **dưới trái**.
- **Mức 2 (vừa):** Bài 2: `count(7)` = `1,3,5,6,7` = 5; `count(10)` = 5; `count(11)` = 6 → nhỏ nhất `x` có `count(x) >= 6` là `11`. Bài 4: heap ban đầu chứa `(matrix[r][0], r, 0)`.
- **Mức 3 (sâu):** Bài 5: `lo = 1`, `hi = m * n`; `while lo < hi: mid = (lo+hi)//2; nếu count(mid) >= k thì hi = mid, ngược lại lo = mid + 1`. Có thể dùng bảng nhỏ để so với vét cạn `sorted(i*j ...)`.

## 19. Đáp án tham khảo cho bài tập

In [32]:
# Bài 1
assert bisect.bisect_right([2, 4, 4, 7], 4) == 3
assert count_le([[1, 2], [3, 4]], 2) == 2
assert (-7 + 2) // 2 == -3
assert Solution().kthSmallest([[1, 2], [1, 3]], 3) == 2
h = [(5, 0), (3, 1), (3, 0)]
heapq.heapify(h)
assert heapq.heappop(h) == (3, 0)
print("Bài 1 OK")

# Bài 2
m = [[1, 3, 5], [6, 7, 12], [11, 14, 14]]
assert (count_le(m, 7), count_le(m, 10), count_le(m, 11)) == (5, 5, 6)
assert Solution().kthSmallest(m, 6) == 11
print("Bài 2 OK")

# Bài 3
def count_todo_ans(matrix, x):
    n = len(matrix)
    r, c, dem = n - 1, 0, 0
    while r >= 0 and c < n:
        if matrix[r][c] <= x:
            dem += r + 1
            c += 1
        else:
            r -= 1
    return dem

rng = random.Random(5)
for _ in range(100):
    n = rng.randint(1, 6)
    mm = tao_ma_tran(n, rng, buoc=3, goc=0)
    for x in range(-1, mm[-1][-1] + 2):
        assert count_todo_ans(mm, x) == count_slow(mm, x)
print("Bài 3 OK")

# Bài 4
def kth_heap2(matrix, k):
    n = len(matrix)
    heap = [(matrix[r][0], r, 0) for r in range(min(n, k))]
    heapq.heapify(heap)
    for _ in range(k - 1):
        val, r, c = heapq.heappop(heap)
        if c + 1 < n:
            heapq.heappush(heap, (matrix[r][c + 1], r, c + 1))
    return heap[0][0]

assert kth_heap2([[1, 5, 9], [10, 11, 13], [12, 13, 15]], 8) == 13
for _ in range(100):
    n = rng.randint(1, 6)
    mm = tao_ma_tran(n, rng, buoc=4, goc=0)
    k = rng.randint(1, n * n)
    assert kth_heap2(mm, k) == kth_sort(mm, k)
print("Bài 4 OK")

# Bài 5
def kth_mul_table(m, n, k):
    def count(x):
        return sum(min(x // i, n) for i in range(1, m + 1))
    lo, hi = 1, m * n
    while lo < hi:
        mid = (lo + hi) // 2
        if count(mid) >= k:
            hi = mid
        else:
            lo = mid + 1
    return lo

assert kth_mul_table(3, 3, 5) == 3 and kth_mul_table(2, 3, 6) == 6
for m_ in range(1, 8):
    for n_ in range(1, 8):
        vals = sorted(i * j for i in range(1, m_ + 1) for j in range(1, n_ + 1))
        for k in range(1, m_ * n_ + 1):
            assert kth_mul_table(m_, n_, k) == vals[k - 1]
print("Bài 5 OK")

Bài 1 OK
Bài 2 OK
Bài 3 OK
Bài 4 OK
Bài 5 OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template (binary search trên giá trị + đếm bậc thang):**

```python
n = len(matrix)

def count_le(x):
    r, c, dem = n - 1, 0, 0
    while r >= 0 and c < n:
        if matrix[r][c] <= x:
            dem += r + 1
            c += 1
        else:
            r -= 1
    return dem

lo, hi = matrix[0][0], matrix[-1][-1]
while lo < hi:
    mid = (lo + hi) // 2
    if count_le(mid) >= k:
        hi = mid
    else:
        lo = mid + 1
return lo
```

**Template (min-heap trộn hàng):**

```python
heap = [(matrix[r][0], r, 0) for r in range(min(n, k))]
heapq.heapify(heap)
for _ in range(k - 1):
    val, r, c = heapq.heappop(heap)
    if c + 1 < n:
        heapq.heappush(heap, (matrix[r][c + 1], r, c + 1))
return heap[0][0]
```

**Checklist:**

- [ ] Đếm `count(mid)` dùng `<=` và so `>= k`; cập nhật `hi = mid` / `lo = mid + 1`.
- [ ] Vòng lặp `while lo < hi` (không `<=`) và trả `lo`.
- [ ] Trung điểm `(lo + hi) // 2` (đúng với số âm trong Python).
- [ ] Không binary search trên chỉ số trải phẳng.
- [ ] Heap: chỉ đẩy phần tử đầu mỗi hàng, nhớ đẩy phần tử kế tiếp sau mỗi `heappop`.
- [ ] Tính cả giá trị trùng (không phải "k-th distinct").
- [ ] Thử `n = 1`, `k = 1`, `k = n²`, ma trận toàn bằng nhau, ma trận có số âm.
- [ ] Đúng signature `class Solution` / `kthSmallest(self, matrix: List[List[int]], k: int) -> int`.